# Projeto Prático: Machine Learning & Inteligência de Mercado
## Análise Estratégica da Concentração no Comércio Global de Bens Criativos (Dataset OpenFCS)

---

> **Componente Curricular:** Machine Learning aplicado à Administração
> **Instituição:** Curso de Graduação em Administração
> **Objetivo:** Aplicação prática de Ciência de Dados, Machine Learning e Inteligência Artificial Generativa para diagnosticar padrões de concentração e (re)configuração competitiva no comércio mundial de bens criativos, a partir do acervo aberto OpenFCS (UNCTAD, alinhado ao UNESCO Framework for Cultural Statistics 2025).

---

### Corpo Docente & Contato

| Atributo | Detalhes |
| :--- | :--- |
| **Professor** | **Sérgio Assunção Monteiro, D.Sc.** |
| **Conecte-se no LinkedIn** | [🌐 linkedin.com/in/sergio-assunção-monteiro](https://www.linkedin.com/in/sergio-assun%C3%A7%C3%A3o-monteiro-b781897b/) |
| **Currículo Lattes** | [🔬 lattes.cnpq.br/9489191035734025](http://lattes.cnpq.br/9489191035734025) |
| **Repositório GitHub** | [💻 github.com/sergiomonteiro76](https://github.com/sergiomonteiro76) |

---

### Sobre este Notebook
Este ambiente foi configurado para que os alunos atuem como **Analistas de Inteligência de Mercado**. Ao longo do semestre, com apoio de modelos de linguagem (IA) integrados ao ecossistema do Google Colab, vamos reconstruir — do dado bruto ao modelo preditivo — o diagnóstico de estrutura competitiva de um setor econômico real: o comércio internacional de bens criativos.

* **Fonte de dados:** [OpenFCS Dataset](https://doi.org/10.5281/zenodo.21211053) — Monteiro & Dubeux (2026), CC-BY-4.0.
* **Material de apoio:** Capítulos 1 a 3 das notas de aula (Ambiente e primeiro contato; Python/pandas/NumPy; Bases de Dados e SQL).

## Aula 7 — Preparação para Modelagem: Encoding, Normalização e Divisão de Dados
Até aqui descrevemos e testamos o dado. A partir de hoje ele precisa virar `X` (features) e `y` (alvo), prontos para um algoritmo de aprendizado. Vamos construir um pipeline de pré-processamento reutilizável — o mesmo que vai reaparecer nas Aulas 8 a 11.

Recarregar o acervo

In [1]:
import requests, zipfile, io, os
import pandas as pd
import numpy as np

url = (
    "https://zenodo.org/records/21211053/files/"
    "openfcs_v1.0.0.zip?download=1"
)
resp = requests.get(url)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall("openfcs")

endereco = "openfcs/openfcs-1.0.0/data/derived/"
edges = pd.read_csv(endereco + "trade_edges.csv")
edges7 = edges[edges["resolution"] == "cer7"].copy()

print(f"edges7: {len(edges7):,} linhas")   # deve dar 1.026.400

edges7: 1,026,400 linhas


### 7.1 Da tabela ao problema de ML: qual é a linha, qual é o alvo?
Vamos reaproveitar a agregação da Aula 6 (economia × domínio × ano) — é o nível de granularidade certo para um primeiro modelo: nem tão fino quanto uma linha bilateral individual (cardinalidade explosiva), nem tão agregado quanto o HHI (perderíamos a variação entre economias).

In [2]:
tabela_ml = (
    edges7.groupby(["fcs_domain", "year", "economy"])["value_usd_millions"]
    .sum()
    .reset_index(name="total_economia")
)
tabela_ml["log_total"] = np.log1p(tabela_ml["total_economia"])

print(f"tabela_ml: {len(tabela_ml):,} linhas")
tabela_ml.head()

tabela_ml: 19,487 linhas


,fcs_domain,year,economy,total_economia,log_total
0,A. Cultural and natural heritage,2002,Andorra,0.002,0.001998
1,A. Cultural and natural heritage,2002,Anguilla,0.013,0.012916
2,A. Cultural and natural heritage,2002,Argentina,0.085,0.081580
3,A. Cultural and natural heritage,2002,Armenia,0.000,0.000000
4,A. Cultural and natural heritage,2002,Australia,13.718,2.689071


### 7.2 Codificação de variáveis categóricas: o problema da cardinalidade
`fcs_domain` tem poucas categorias — one-hot encoding é seguro. `economy` tem mais de 200 — one-hot geraria centenas de colunas esparsas. Vamos ver o tamanho do problema antes de escolher a solução.

In [3]:
print("Cardinalidade das variaveis categoricas:")
print(f"fcs_domain: {tabela_ml['fcs_domain'].nunique()} categorias")
print(f"economy:    {tabela_ml['economy'].nunique()} categorias")

# One-hot em fcs_domain (baixa cardinalidade, seguro)
dominio_onehot = pd.get_dummies(tabela_ml["fcs_domain"], prefix="dom")
print(f"\nColunas geradas por one-hot em fcs_domain: {dominio_onehot.shape[1]}")

# E se fizessemos o mesmo com economy?
economia_onehot = pd.get_dummies(tabela_ml["economy"], prefix="eco")
print(f"Colunas que seriam geradas em economy:      {economia_onehot.shape[1]}")

Cardinalidade das variaveis categoricas:
fcs_domain: 6 categorias
economy:    204 categorias

Colunas geradas por one-hot em fcs_domain: 6
Colunas que seriam geradas em economy:      204


### 7.3 Uma alternativa ao one-hot: codificação por frequência
Em vez de uma coluna binária por país, substituímos cada economia pelo seu total histórico exportado — uma única coluna numérica, sem explosão de dimensionalidade. É uma forma simples de *target/frequency encoding*.

In [4]:
freq_economia = tabela_ml.groupby("economy")["total_economia"].sum()
tabela_ml["economy_freq"] = tabela_ml["economy"].map(freq_economia)

tabela_ml[["economy", "economy_freq"]].drop_duplicates().sort_values(
    "economy_freq", ascending=False
).head()

,economy,economy_freq
39,G-77 (Group of 77),4922947.616
20,China,3987058.568
109,United States,994654.104
21,"China, Hong Kong SAR",847100.481
54,Italy,762683.896


### 7.4 Normalização e padronização
`year` varia entre ~2002 e 2024; `economy_freq` varia em milhões. Algoritmos sensíveis a escala (regressão regularizada, KNN, redes neurais) tratariam `economy_freq` como "mais importante" só por ter números maiores. O `StandardScaler` resolve isso.

In [5]:
from sklearn.preprocessing import StandardScaler

colunas_numericas = ["year", "economy_freq"]
scaler = StandardScaler()

tabela_ml_scaled = tabela_ml.copy()
tabela_ml_scaled[colunas_numericas] = scaler.fit_transform(
    tabela_ml[colunas_numericas]
)

print("Antes (media, desvio):")
print(tabela_ml[colunas_numericas].agg(["mean", "std"]))
print("\nDepois (media ~0, desvio ~1):")
print(tabela_ml_scaled[colunas_numericas].agg(["mean", "std"]))

Antes (media, desvio):
             year   economy_freq
mean  2013.003079  131817.415954
std      6.550649  546289.658095

Depois (media ~0, desvio ~1):
              year  economy_freq
mean -1.039616e-14 -1.458496e-18
std   1.000026e+00  1.000026e+00


### 7.5 Divisão treino/teste: aleatória vs. temporal

In [6]:
from sklearn.model_selection import train_test_split

X = tabela_ml[["fcs_domain", "year", "economy_freq"]]
y = tabela_ml["log_total"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42
)

anos_treino = sorted(X_treino["year"].unique())
anos_teste = sorted(X_teste["year"].unique())
print(f"Treino: {len(X_treino):,} linhas | anos {anos_treino[0]}-{anos_treino[-1]}")
print(f"Teste:  {len(X_teste):,} linhas | anos {anos_teste[0]}-{anos_teste[-1]}")

Treino: 15,589 linhas | anos 2002-2024
Teste:  3,898 linhas | anos 2002-2024


**split temporal**

In [7]:
corte = 2020

treino_temporal = tabela_ml[tabela_ml["year"] < corte]
teste_temporal = tabela_ml[tabela_ml["year"] >= corte]

print(f"Treino (< {corte}):  {len(treino_temporal):,} linhas")
print(f"Teste  (>= {corte}): {len(teste_temporal):,} linhas")

Treino (< 2020):  15,374 linhas
Teste  (>= 2020): 4,113 linhas


**⚠️ Atenção:** o split aleatório da Célula 12 mistura anos livremente — o modelo pode "treinar" com 2023 e ser testado em 2015. Em dados de painel temporal, isso é vazamento (*leakage*): na vida real, você nunca tem acesso ao futuro para prever o passado. Para uma avaliação honesta de capacidade preditiva, o corte temporal da Célula 13 é mais defensável. Guarde essa ideia — ela volta com força na Aula 12 (séries temporais).

### 7.6 Validação cruzada (k-fold)

In [8]:
from sklearn.model_selection import KFold

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

for i, (idx_treino, idx_val) in enumerate(kfold.split(X_treino)):
    print(f"Fold {i+1}: {len(idx_treino):,} treino / {len(idx_val):,} validacao")

Fold 1: 12,471 treino / 3,118 validacao
Fold 2: 12,471 treino / 3,118 validacao
Fold 3: 12,471 treino / 3,118 validacao
Fold 4: 12,471 treino / 3,118 validacao
Fold 5: 12,472 treino / 3,117 validacao


### 7.7 Um pipeline reutilizável
Reunindo tudo: `ColumnTransformer` aplica a transformação certa a cada coluna (one-hot em `fcs_domain`, padronização nas numéricas), e o `Pipeline` empacota isso como um único objeto — que volta a ser usado, sem reescrever nada, nas Aulas 8 a 11.

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

pre_processador = ColumnTransformer(
    transformers=[
        ("categorica", OneHotEncoder(handle_unknown="ignore"), ["fcs_domain"]),
        ("numerica", StandardScaler(), ["year", "economy_freq"]),
    ]
)

pipeline = Pipeline(steps=[("preprocessamento", pre_processador)])

X_treino_transformado = pipeline.fit_transform(X_treino)
print(f"Formato de X_treino antes do pipeline: {X_treino.shape}")
print(f"Formato de X_treino depois do pipeline: {X_treino_transformado.shape}")

Formato de X_treino antes do pipeline: (15589, 3)
Formato de X_treino depois do pipeline: (15589, 8)


O pipeline aprendeu (`fit`) as categorias e a escala **apenas no treino** — é isso que garante que nenhuma informação do teste "vaze" para o pré-processamento. Na Aula 9, este mesmo `pipeline` vai anteceder um `LinearRegression`; na Aula 10, um classificador. Nada aqui precisará ser reescrito.

## 🧪 Exercícios Práticos — Aula 7

> **Como usar:** resolva cada exercício em uma célula de código abaixo do enunciado. Depois, leve o resultado para uma IA usando o *prompt sugerido*.

---

### Exercício 1 — Cardinalidade de `partner`
**📝 Tarefa:** Calcule a cardinalidade de `partner` em `edges7` (não em `tabela_ml`). Ela é parecida com a de `economy`? Que estratégia de encoding você recomendaria para essa coluna, e por quê?

**🤖 Pergunte à IA:**
> "Uma coluna categórica tem [N] categorias distintas em um dataset com [M] linhas. Quais são as vantagens e desvantagens de usar one-hot encoding, frequency encoding e target encoding nesse cenário?"

---

### Exercício 2 — StandardScaler vs. MinMaxScaler
**📝 Tarefa:** Aplique `MinMaxScaler` (em vez de `StandardScaler`) nas mesmas colunas da Célula 10. Compare os valores resultantes. Em que situação um seria preferível ao outro?

**🤖 Pergunte à IA:**
> "Apliquei StandardScaler e MinMaxScaler na mesma coluna numérica e os resultados foram: [descreva]. Em que tipo de algoritmo de machine learning a escolha entre os dois realmente importa?"

---

### Exercício 3 — Um corte temporal diferente
**📝 Tarefa:** Repita o split temporal da Célula 13 usando `corte = 2015` em vez de 2020. Como o tamanho relativo de treino e teste muda? Qual corte parece mais equilibrado?

**🤖 Pergunte à IA:**
> "Testei dois cortes temporais para dividir treino e teste: [corte 1] resultou em [X]/[Y] linhas, [corte 2] resultou em [X2]/[Y2]. Que critérios, além do tamanho, eu deveria considerar para escolher o corte certo?"

---

### Exercício 4 — Vazamento temporal, na prática
**📝 Tarefa:** No split aleatório da Célula 12, verifique quantas linhas de `X_treino` têm `year >= 2020` (ou seja, anos que também aparecem no teste temporal da Célula 13). O que isso implica se você tivesse usado o split aleatório para simular uma previsão real de 2020 em diante?

**🤖 Pergunte à IA:**
> "Descobri que meu split aleatório de treino contém [N] linhas de anos que eu pretendia usar só para teste temporal. Por que isso é um problema mesmo que a acurácia do modelo pareça boa?"

---

### Exercício 5 — Expandindo o pipeline
**📝 Tarefa:** Modifique o `ColumnTransformer` da Célula 17 para incluir também `economy` diretamente (não `economy_freq`), usando `OneHotEncoder`. Compare o número de colunas resultante com a versão que usa `economy_freq`.

**🤖 Pergunte à IA:**
> "Comparei duas versões do meu pipeline: uma com one-hot em uma coluna de alta cardinalidade, outra com frequency encoding na mesma coluna. As formas resultantes (shapes) foram [X] e [Y]. Quais os prós e contras de cada abordagem para um modelo de regressão linear?"